# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR^2 dataset using the `mlcroissant` library, focusing on working with Croissant schemas and referencing all elements by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (as a single object, not as a dictionary)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets (tables), and their related field and column `@id`s using the Croissant schema.

In [ ]:
# Retrieve all record sets and their fields by @id
print('Available record sets (by @id):')
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name','(no name)')}")
    print("  Fields (columns):")
    if 'field' in record_set:
        for field in record_set['field']:
            if isinstance(field, dict):
                print(f"    - @id: {field['@id']}\tName: {field.get('name','(no name)')}")
            else:
                print(f"    - @id: {field}")
    print("")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for further analysis. Use the record set and field `@id`s as shown above.

In [ ]:
# List all RecordSet @ids
available_record_set_ids = [rs['@id'] for rs in metadata.record_sets]

# Choose a record set for analysis (typically main table, here we pick the first record set)
main_record_set_id = available_record_set_ids[0]
print(f"Using record set: {main_record_set_id}")

# Load records for each record set
dataframes = {}
for record_set_id in available_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}")

# Display columns of the main DataFrame
main_columns = dataframes[main_record_set_id].columns.tolist()
print(f"Columns for main record set: {main_columns}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filtering by numeric fields, normalization, and grouping by a categorical field. All fields are referenced by their `@id`. Ensure the field exists in the dataframe.

In [ ]:
# List all numeric fields (by @id and name, if available)
main_df = dataframes[main_record_set_id]

print('Numeric columns available:')
numeric_cols = main_df.select_dtypes(include=['number']).columns.tolist()
print(numeric_cols)

# Pick a numeric field for filtering (by @id string)
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # Use the first numeric column (usually its @id)
    threshold = main_df[numeric_field_id].quantile(0.75) # set threshold at 75th percentile for demo
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    
    # Normalize selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another (non-numeric) column
    non_numeric_cols = [col for col in main_df.columns if main_df[col].dtype == 'object' and col != numeric_field_id]
    if non_numeric_cols:
        group_field_id = non_numeric_cols[0]
        grouped_df = filtered_df.groupby(group_field_id, observed=True).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (showing group means):")
        print(grouped_df.head())
else:
    print("No numeric fields found for filtering and analysis.")

## 5. Visualization
Visualize the distribution of a selected numeric field, and relationships between two fields if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualization if a numeric field is available
if numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Optionally plot relationship with another numeric field if available
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=main_df[numeric_cols[0]], y=main_df[numeric_cols[1]])
        plt.title(f'{numeric_cols[0]} vs {numeric_cols[1]}')
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()

## 6. Conclusion
We demonstrated loading, accessing metadata, referencing fields by `@id`, filtering, normalization, grouping, and basic visualization for the FAIR^2 dataset using `mlcroissant`. The dataset is rich in protein modification and abundance data, accessible programmatically for customized downstream analysis.

Continue exploring by modifying the record set or fields referenced by `@id` to match your research questions.